# 00 Setup: build the whole AdventureWorks demo

Import this one notebook into a Fabric workspace and select **Run all**. You do not need to attach a Lakehouse. BEST DAY EVER!!!

It builds everything, in this order:

1. A schema-enabled Lakehouse (`AdventureWorksLH`).
2. **Bronze**: the 23 AdventureWorks files, downloaded from GitHub into `Files/AdventureWorks`.
3. **Silver**: typed, validated Delta tables (runs the `AW_01_Load_Silver` notebook).
4. **Gold**: a star schema with current dates (runs the `AW_02_Build_Gold` notebook).
5. A Direct Lake semantic model with measures and row-level security roles, and a report on top of it.
6. A pipeline that rebuilds the tables, scheduled daily so the dates always end today.

It is safe to run again: existing items are updated, not duplicated.

**What you need:** a workspace on a Fabric capacity (a trial works), the Contributor role or higher, and outbound internet access from the notebook to reach GitHub.

**Daily schedule:** step 6 turns on a daily pipeline run, which uses a small amount of capacity every day. Set `SCHEDULE_DAILY_REFRESH = False` below if you do not want that.

In [ ]:
# Settings. Change these before you run, if you need to.
REPO = "KratosDataGod/FabricAdventureWorksLakehouse"   # use your own fork here if you have one
BRANCH = "main"
LAKEHOUSE_NAME = "AdventureWorksLH"

DEPLOY_POWER_BI = True             # semantic model and report
DEPLOY_PIPELINE = True             # pipeline that reloads silver and rebuilds gold
SCHEDULE_DAILY_REFRESH = True      # run that pipeline every day
DAILY_REFRESH_TIME_UTC = "06:00"

In [ ]:
import base64
import io
import json
import os
import shutil
import time
import zipfile
from datetime import datetime, timedelta, timezone

import notebookutils
import requests

API = "https://api.fabric.microsoft.com/v1"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]

SILVER_NOTEBOOK = "AW_01_Load_Silver"
GOLD_NOTEBOOK = "AW_02_Build_Gold"
MODEL_NAME = "AdventureWorks Sales"
REPORT_NAME = "AdventureWorks Sales Overview"
PIPELINE_NAME = "AdventureWorks Daily Refresh"


def call(method, url, body=None):
    """Call the Fabric REST API. Waits for long running operations and returns the JSON result."""
    full_url = url if url.startswith("http") else API + url
    while True:
        headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
        response = requests.request(method, full_url, headers=headers, json=body)
        if response.status_code != 429:
            break
        time.sleep(int(response.headers.get("Retry-After", 10)))   # throttled: wait and try again
    if response.status_code >= 400:
        raise Exception(f"{method} {full_url} failed with {response.status_code}: {response.text}")
    if response.status_code == 202 and "x-ms-operation-id" in response.headers:
        operation = f"{API}/operations/{response.headers['x-ms-operation-id']}"
        while True:
            time.sleep(3)
            state = requests.get(operation, headers=headers).json()
            if state.get("status") in ("Succeeded", "Failed"):
                break
        if state["status"] == "Failed":
            raise Exception(f"{method} {full_url} failed: {json.dumps(state.get('error'))}")
        result = requests.get(f"{operation}/result", headers=headers)
        return result.json() if result.status_code == 200 and result.content else {}
    return response.json() if response.content else {}


def find_item(item_type, name):
    """Return the workspace item with this type and name, or None."""
    url = f"/workspaces/{WORKSPACE_ID}/items?type={item_type}"
    while url:
        page = call("GET", url)
        for item in page.get("value", []):
            if item["displayName"] == name:
                return item
        url = page.get("continuationUri")
    return None


def deploy_item(item_type, name, files, definition_format=None):
    """Create the item from its definition files, or update it if it already exists. Returns the item ID."""
    definition = {"parts": [
        {"path": path, "payload": base64.b64encode(content).decode(), "payloadType": "InlineBase64"}
        for path, content in sorted(files.items())
    ]}
    if definition_format:
        definition["format"] = definition_format
    existing = find_item(item_type, name)
    if existing:
        call("POST", f"/workspaces/{WORKSPACE_ID}/items/{existing['id']}/updateDefinition", {"definition": definition})
        print(f"Updated {item_type} '{name}'")
        return existing["id"]
    created = call("POST", f"/workspaces/{WORKSPACE_ID}/items", {"displayName": name, "type": item_type, "definition": definition})
    print(f"Created {item_type} '{name}'")
    return created.get("id") or find_item(item_type, name)["id"]


def read_folder(folder):
    """Read every file in a folder as {relative path: bytes}, skipping local-only files."""
    files = {}
    for directory, _, names in os.walk(folder):
        for file_name in names:
            path = os.path.relpath(os.path.join(directory, file_name), folder).replace(os.sep, "/")
            if path == ".platform" or path.startswith(".pbi/"):
                continue
            with open(os.path.join(directory, file_name), "rb") as handle:
                files[path] = handle.read()
    return files

## 1. Get the repo and create the Lakehouse

In [ ]:
# Download one consistent copy of the repo: data files, notebooks, semantic model, report and pipeline.
download = requests.get(f"https://github.com/{REPO}/archive/refs/heads/{BRANCH}.zip", timeout=300)
download.raise_for_status()
shutil.rmtree("/tmp/aw_repo", ignore_errors=True)
zipfile.ZipFile(io.BytesIO(download.content)).extractall("/tmp/aw_repo")
REPO_DIR = os.path.join("/tmp/aw_repo", os.listdir("/tmp/aw_repo")[0])
print(f"Downloaded {REPO}@{BRANCH}: {len(os.listdir(os.path.join(REPO_DIR, 'AdventureWorks')))} data files")

In [ ]:
existing_lakehouses = [item.displayName for item in notebookutils.lakehouse.list()]
if LAKEHOUSE_NAME in existing_lakehouses:
    lakehouse = notebookutils.lakehouse.get(LAKEHOUSE_NAME)
    print(f"Using existing Lakehouse '{LAKEHOUSE_NAME}'")
else:
    lakehouse = notebookutils.lakehouse.create(LAKEHOUSE_NAME, "AdventureWorks demo: bronze files, silver and gold tables", {"enableSchemas": True})
    print(f"Created Lakehouse '{LAKEHOUSE_NAME}'")
LAKEHOUSE_ID = lakehouse.id

properties = call("GET", f"/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}").get("properties", {})
if "defaultSchema" not in properties:
    raise Exception(
        f"The Lakehouse '{LAKEHOUSE_NAME}' was created without schemas, so it cannot hold the silver and gold schemas. "
        "Set LAKEHOUSE_NAME to a new name and run again, and this notebook will create a schema-enabled Lakehouse for you."
    )

ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"

## 2. Bronze: land the files

In [ ]:
landing = f"{ROOT}/Files/AdventureWorks"
if notebookutils.fs.exists(landing):
    notebookutils.fs.rm(landing, True)   # start clean so a rerun never leaves old files behind
notebookutils.fs.cp(f"file:{REPO_DIR}/AdventureWorks", landing, True)
print(f"Copied {len(notebookutils.fs.ls(landing))} files to Files/AdventureWorks")

## 3 and 4. Silver and gold: import the two notebooks and run them

In [ ]:
def import_notebook(file_name, display_name):
    with open(os.path.join(REPO_DIR, "NoteBooks", file_name), "rb") as handle:
        return deploy_item("Notebook", display_name, {"notebook-content.ipynb": handle.read()}, "ipynb")


def run_notebook(name):
    """Run a notebook in this workspace. A notebook that was just created can take a moment to become available."""
    for attempt in range(6):
        try:
            return notebookutils.notebook.run(name, 3600, {"LAKEHOUSE_NAME": LAKEHOUSE_NAME})
        except Exception as error:
            if attempt == 5 or "not found" not in str(error).lower():
                raise
            time.sleep(20)


silver_notebook_id = import_notebook("01_Load_Silver.ipynb", SILVER_NOTEBOOK)
gold_notebook_id = import_notebook("02_Build_Gold.ipynb", GOLD_NOTEBOOK)

run_notebook(SILVER_NOTEBOOK)
print("Silver tables loaded and validated")
run_notebook(GOLD_NOTEBOOK)
print("Gold tables built and validated")

## 5. Semantic model and report

The model uses Direct Lake on OneLake, so it reads the gold Delta tables directly: no import, no refresh schedule.

In [ ]:
model_id = report_id = None
if DEPLOY_POWER_BI:
    # Point the model at this workspace and Lakehouse.
    model_files = read_folder(os.path.join(REPO_DIR, "PowerBI", "AdventureWorks.SemanticModel"))
    expressions = model_files["definition/expressions.tmdl"].decode("utf-8")
    expressions = expressions.replace("00000000-0000-0000-0000-000000000000", WORKSPACE_ID).replace("11111111-1111-1111-1111-111111111111", LAKEHOUSE_ID)
    model_files["definition/expressions.tmdl"] = expressions.encode("utf-8")
    model_id = deploy_item("SemanticModel", MODEL_NAME, model_files)

    # Load the model once so the report has data straight away. After this, Direct Lake keeps itself up to date.
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    refreshes = f"https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets/{model_id}/refreshes"
    started = requests.post(refreshes, headers=headers, json={"notifyOption": "NoNotification"})
    status = "Unknown" if started.status_code == 202 else f"not started ({started.status_code}: {started.text})"
    while status == "Unknown":   # the API reports a running refresh as "Unknown"
        time.sleep(5)
        latest = requests.get(f"{refreshes}?$top=1", headers=headers).json().get("value", [{}])[0]
        status = latest.get("status", "Unknown")
    if status == "Completed":
        print("Semantic model loaded")
    else:
        print(f"The semantic model was deployed but its first refresh did not complete: {status}. "
              f"Open '{MODEL_NAME}' in the workspace and select Refresh to see the reason.")

    # Point the report at the model that was just deployed.
    report_files = read_folder(os.path.join(REPO_DIR, "PowerBI", "AdventureWorks.Report"))
    report_files["definition.pbir"] = json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definitionProperties/2.0.0/schema.json",
        "version": "4.0",
        "datasetReference": {"byConnection": {"connectionString": f"semanticmodelid={model_id}"}},
    }, indent=2).encode("utf-8")
    report_id = deploy_item("Report", REPORT_NAME, report_files)

## 6. Daily refresh pipeline

In [ ]:
if DEPLOY_PIPELINE:
    with open(os.path.join(REPO_DIR, "Pipeline", "pipeline-content.json"), encoding="utf-8") as handle:
        pipeline = handle.read()
    pipeline = (pipeline
                .replace("00000000-0000-0000-0000-000000000000", WORKSPACE_ID)
                .replace("22222222-2222-2222-2222-222222222222", silver_notebook_id)
                .replace("33333333-3333-3333-3333-333333333333", gold_notebook_id)
                .replace("AdventureWorksLH", LAKEHOUSE_NAME))
    pipeline_id = deploy_item("DataPipeline", PIPELINE_NAME, {"pipeline-content.json": pipeline.encode("utf-8")})

    if SCHEDULE_DAILY_REFRESH:
        schedules = f"/workspaces/{WORKSPACE_ID}/items/{pipeline_id}/jobs/Pipeline/schedules"
        if call("GET", schedules).get("value"):
            print("The pipeline already has a schedule; left as it is")
        else:
            tomorrow = (datetime.now(timezone.utc) + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
            call("POST", schedules, {
                "enabled": True,
                "configuration": {
                    "type": "Daily",
                    "startDateTime": tomorrow.strftime("%Y-%m-%dT%H:%M:%S"),
                    "endDateTime": (tomorrow + timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%S"),
                    "localTimeZoneId": "UTC",
                    "times": [DAILY_REFRESH_TIME_UTC],
                },
            })
            print(f"Scheduled '{PIPELINE_NAME}' daily at {DAILY_REFRESH_TIME_UTC} UTC for one year")

## Done

In [ ]:
print("Setup complete. BEST DAY EVER!!!")
print(f"Lakehouse: https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}")
if report_id:
    print(f"Report:    https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/reports/{report_id}")